# BFP Health Monitoring: Vibration & Bearing Temperature Anomaly Detection

From the [Sisyphean Gridworks ML Playground](https://sgridworks.com/ml-playground/guides/17-bfp-health-monitoring.html)

## Setup

Clone the repository and install dependencies. Run this cell first.

In [ ]:
import os, subprocess

# Colab: clone the repo and cd into it
# Local: detect if we are already inside the repo
if not os.path.exists('sisyphean-power-and-light'):
    if os.path.exists('../sisyphean-power-and-light'):
        os.chdir('..')  # running from notebooks/ subfolder
    else:
        subprocess.run(['git', 'clone', 'https://github.com/SGridworks/Dynamic-Network-Model.git'], capture_output=True)
        os.chdir('Dynamic-Network-Model')

print(f'Working directory: {os.getcwd()}')
# !pip install -q pandas numpy matplotlib seaborn scikit-learn pyarrow


## Step 0: Verify Setup

Load the BFP hourly parquet file and confirm the expected row count (8,784 rows for a full leap year of hourly data).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Load BFP hourly data
DATA_PATH = "sisyphean-power-and-light/generation/timeseries/bfp_train_hourly.parquet"
df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp").sort_index()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
assert len(df) == 8784, f"Expected 8,784 rows, got {len(df)}"
print("\nSetup verified.")

## Step 1: Load and Identify Health-Critical Columns

Boiler feed pumps (BFPs) are the highest-power rotating equipment in the feedwater system. The two primary health indicators are **bearing temperatures** (journal and thrust bearings at both drive-end and non-drive-end) and **shaft vibration** (X and Y proximity probes at DE and NDE). We also track seal leakage and axial displacement as secondary indicators.

In [ ]:
# Health-critical tags for both pumps
health_tags = {
    "BFP-A": {
        "brg_de":   "U1_BFPA_BRG_DE_TEMP",
        "brg_nde":  "U1_BFPA_BRG_NDE_TEMP",
        "thr_act":  "U1_BFPA_THR_ACT_TEMP",
        "vib_de_x": "U1_BFPA_VIB_DE_X",
        "vib_nde_x":"U1_BFPA_VIB_NDE_X",
        "seal_leak":"U1_BFPA_SEAL_DE_LEAK",
        "axial":    "U1_BFPA_AXIAL_DISP",
        "run":      "U1_BFPA_RUN_STATUS",
    },
    "BFP-B": {
        "brg_de":   "U1_BFPB_BRG_DE_TEMP",
        "brg_nde":  "U1_BFPB_BRG_NDE_TEMP",
        "thr_act":  "U1_BFPB_THR_ACT_TEMP",
        "vib_de_x": "U1_BFPB_VIB_DE_X",
        "vib_nde_x":"U1_BFPB_VIB_NDE_X",
        "seal_leak":"U1_BFPB_SEAL_DE_LEAK",
        "axial":    "U1_BFPB_AXIAL_DISP",
        "run":      "U1_BFPB_RUN_STATUS",
    },
}

# Quick summary of health tags
for pump, tags in health_tags.items():
    running_hours = (df[tags["run"]] > 0).sum()
    print(f"\n{pump}:")
    print(f"  Running hours: {running_hours:,}")
    for name, col in tags.items():
        if name == "run":
            continue
        running_data = df.loc[df[tags["run"]] > 0, col]
        if len(running_data) > 0:
            print(f"  {name:10s}: mean={running_data.mean():.1f}  "
                  f"min={running_data.min():.1f}  max={running_data.max():.1f}")

## Step 2: Explore Bearing Temperatures and Vibration Over Time

Plot the raw time series for bearing temperatures and vibration for both pumps. This gives us a visual baseline before we start computing statistics. Look for trends, spikes, and periods where values drift upward -- these are potential fault indicators.

In [ ]:
# Bearing temperature trends for both pumps
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

# BFP-A bearing temps (only when running)
mask_a = df["U1_BFPA_RUN_STATUS"] > 0
axes[0].plot(df.index[mask_a], df.loc[mask_a, "U1_BFPA_BRG_DE_TEMP"],
             color="#5FCCDB", linewidth=0.5, label="DE Bearing")
axes[0].plot(df.index[mask_a], df.loc[mask_a, "U1_BFPA_BRG_NDE_TEMP"],
             color="#2D6A7A", linewidth=0.5, label="NDE Bearing")
axes[0].plot(df.index[mask_a], df.loc[mask_a, "U1_BFPA_THR_ACT_TEMP"],
             color="#D69E2E", linewidth=0.5, label="Thrust Active")
axes[0].set_ylabel("Temperature (degC)")
axes[0].set_title("BFP-A Bearing Temperatures")
axes[0].legend(fontsize=8)

# BFP-B bearing temps (only when running)
mask_b = df["U1_BFPB_RUN_STATUS"] > 0
axes[1].plot(df.index[mask_b], df.loc[mask_b, "U1_BFPB_BRG_DE_TEMP"],
             color="#5FCCDB", linewidth=0.5, label="DE Bearing")
axes[1].plot(df.index[mask_b], df.loc[mask_b, "U1_BFPB_BRG_NDE_TEMP"],
             color="#2D6A7A", linewidth=0.5, label="NDE Bearing")
axes[1].plot(df.index[mask_b], df.loc[mask_b, "U1_BFPB_THR_ACT_TEMP"],
             color="#D69E2E", linewidth=0.5, label="Thrust Active")
axes[1].set_ylabel("Temperature (degC)")
axes[1].set_xlabel("Date")
axes[1].set_title("BFP-B Bearing Temperatures")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Vibration trends for both pumps
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(df.index[mask_a], df.loc[mask_a, "U1_BFPA_VIB_DE_X"],
             color="#5FCCDB", linewidth=0.5, label="DE X-probe")
axes[0].plot(df.index[mask_a], df.loc[mask_a, "U1_BFPA_VIB_NDE_X"],
             color="#2D6A7A", linewidth=0.5, label="NDE X-probe")
axes[0].set_ylabel("Vibration (um pk-pk)")
axes[0].set_title("BFP-A Shaft Vibration")
axes[0].legend(fontsize=8)

axes[1].plot(df.index[mask_b], df.loc[mask_b, "U1_BFPB_VIB_DE_X"],
             color="#5FCCDB", linewidth=0.5, label="DE X-probe")
axes[1].plot(df.index[mask_b], df.loc[mask_b, "U1_BFPB_VIB_NDE_X"],
             color="#2D6A7A", linewidth=0.5, label="NDE X-probe")
axes[1].set_ylabel("Vibration (um pk-pk)")
axes[1].set_xlabel("Date")
axes[1].set_title("BFP-B Shaft Vibration")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Step 3: Calculate Rolling Statistics

A single high reading does not necessarily mean trouble. What matters is the **trend**. We compute 24-hour rolling mean and standard deviation for bearing temperatures and vibration. Rising rolling mean indicates developing faults. Rising rolling std indicates increasing instability.

In [ ]:
# Compute 24-hour rolling statistics for key health tags
window = 24  # hours

rolling_cols = {
    "A_brg_de":  "U1_BFPA_BRG_DE_TEMP",
    "A_brg_nde": "U1_BFPA_BRG_NDE_TEMP",
    "A_vib_de":  "U1_BFPA_VIB_DE_X",
    "A_vib_nde": "U1_BFPA_VIB_NDE_X",
    "B_brg_de":  "U1_BFPB_BRG_DE_TEMP",
    "B_brg_nde": "U1_BFPB_BRG_NDE_TEMP",
    "B_vib_de":  "U1_BFPB_VIB_DE_X",
    "B_vib_nde": "U1_BFPB_VIB_NDE_X",
}

for label, col in rolling_cols.items():
    df[f"{label}_roll_mean"] = df[col].rolling(window, min_periods=1).mean()
    df[f"{label}_roll_std"]  = df[col].rolling(window, min_periods=1).std()

# Plot rolling mean for BFP-A bearing DE temp
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(df.index[mask_a], df.loc[mask_a, "U1_BFPA_BRG_DE_TEMP"],
             color="#5FCCDB", linewidth=0.3, alpha=0.5, label="Raw")
axes[0].plot(df.index[mask_a], df.loc[mask_a, "A_brg_de_roll_mean"],
             color="#2D6A7A", linewidth=1.5, label="24h Rolling Mean")
axes[0].set_ylabel("Temperature (degC)")
axes[0].set_title("BFP-A DE Bearing: Raw vs 24h Rolling Mean")
axes[0].legend(fontsize=8)

axes[1].plot(df.index[mask_a], df.loc[mask_a, "A_brg_de_roll_std"],
             color="#D69E2E", linewidth=1.0)
axes[1].set_ylabel("Rolling Std (degC)")
axes[1].set_xlabel("Date")
axes[1].set_title("BFP-A DE Bearing: 24h Rolling Standard Deviation")

plt.tight_layout()
plt.show()

print("Rolling statistics computed for all health tags.")

## Step 4: Threshold-Based Alerts from Design Parameters

Every rotating machine has OEM-specified alarm and trip setpoints. We load these from the `design_parameters.json` file and compare actual readings against them. This is the simplest form of condition monitoring: if a value exceeds a threshold, raise an alert.

In [ ]:
# Load alarm setpoints from design parameters
with open("sisyphean-power-and-light/generation/reference/design_parameters.json") as f:
    design = json.load(f)

setpoints = design["alarm_setpoints"]
print("Alarm setpoints loaded:")
for k, v in setpoints.items():
    print(f"  {k}: {v}")

# Define threshold checks: (column, alarm_threshold, trip_threshold, description)
threshold_checks = [
    ("U1_BFPA_BRG_DE_TEMP",  setpoints["journal_bearing_high_alarm_degc"],
     setpoints["journal_bearing_high_trip_degc"], "BFP-A DE Journal Bearing"),
    ("U1_BFPA_BRG_NDE_TEMP", setpoints["journal_bearing_high_alarm_degc"],
     setpoints["journal_bearing_high_trip_degc"], "BFP-A NDE Journal Bearing"),
    ("U1_BFPA_VIB_DE_X",     setpoints["shaft_vibration_high_alarm_um"],
     setpoints["shaft_vibration_high_trip_um"],   "BFP-A DE Vibration"),
    ("U1_BFPA_VIB_NDE_X",    setpoints["shaft_vibration_high_alarm_um"],
     setpoints["shaft_vibration_high_trip_um"],   "BFP-A NDE Vibration"),
    ("U1_BFPB_BRG_DE_TEMP",  setpoints["journal_bearing_high_alarm_degc"],
     setpoints["journal_bearing_high_trip_degc"], "BFP-B DE Journal Bearing"),
    ("U1_BFPB_BRG_NDE_TEMP", setpoints["journal_bearing_high_alarm_degc"],
     setpoints["journal_bearing_high_trip_degc"], "BFP-B NDE Journal Bearing"),
    ("U1_BFPB_VIB_DE_X",     setpoints["shaft_vibration_high_alarm_um"],
     setpoints["shaft_vibration_high_trip_um"],   "BFP-B DE Vibration"),
    ("U1_BFPB_VIB_NDE_X",    setpoints["shaft_vibration_high_alarm_um"],
     setpoints["shaft_vibration_high_trip_um"],   "BFP-B NDE Vibration"),
]

# Count threshold exceedances
print("\nThreshold exceedance summary (running hours only):")
print(f"{'Sensor':<30s} {'Alarm Hrs':>10s} {'Trip Hrs':>10s}")
print("-" * 52)

for col, alarm, trip, desc in threshold_checks:
    # Determine which pump run status to use
    run_col = "U1_BFPA_RUN_STATUS" if "BFPA" in col else "U1_BFPB_RUN_STATUS"
    running = df[run_col] > 0
    alarm_count = ((df[col] > alarm) & running).sum()
    trip_count  = ((df[col] > trip) & running).sum()
    print(f"{desc:<30s} {alarm_count:>10d} {trip_count:>10d}")

## Step 5: Train an Isolation Forest Anomaly Detector

Threshold-based monitoring catches obvious problems but misses subtle multi-variable patterns. An Isolation Forest learns the normal distribution of bearing temperatures and vibration features jointly, then flags points that are statistically unusual even when no single reading exceeds a threshold.

In [ ]:
# Build feature matrix for BFP-A (running hours only)
feature_cols_a = [
    "U1_BFPA_BRG_DE_TEMP", "U1_BFPA_BRG_NDE_TEMP", "U1_BFPA_THR_ACT_TEMP",
    "U1_BFPA_VIB_DE_X", "U1_BFPA_VIB_NDE_X",
    "U1_BFPA_SEAL_DE_LEAK", "U1_BFPA_AXIAL_DISP",
]

df_a = df.loc[mask_a, feature_cols_a].dropna()
print(f"BFP-A running samples: {len(df_a):,}")

# Train on healthy baseline (Jan-Mar) and score all running data
healthy_mask = df_a.index < "2024-04-01"
X_healthy = df_a.loc[healthy_mask]
print(f"Healthy baseline samples (Jan-Mar): {len(X_healthy):,}")

# Fit scaler on healthy baseline
scaler_a = StandardScaler()
X_healthy_scaled = scaler_a.fit_transform(X_healthy)

# Train Isolation Forest on healthy data only
iso_a = IsolationForest(
    n_estimators=200,
    contamination=0.02,  # expect ~2% anomalies in healthy data (noise floor)
    random_state=42,
)
iso_a.fit(X_healthy_scaled)

# Score ALL running data
X_all_scaled = scaler_a.transform(df_a)
df_a["anomaly_score"] = iso_a.decision_function(X_all_scaled)
df_a["is_anomaly"] = iso_a.predict(X_all_scaled)  # -1 = anomaly, 1 = normal

n_anomalies = (df_a["is_anomaly"] == -1).sum()
print(f"\nAnomalies detected: {n_anomalies} ({n_anomalies/len(df_a)*100:.1f}%)")

In [ ]:
# Repeat for BFP-B
feature_cols_b = [
    "U1_BFPB_BRG_DE_TEMP", "U1_BFPB_BRG_NDE_TEMP", "U1_BFPB_THR_ACT_TEMP",
    "U1_BFPB_VIB_DE_X", "U1_BFPB_VIB_NDE_X",
    "U1_BFPB_SEAL_DE_LEAK", "U1_BFPB_AXIAL_DISP",
]

df_b = df.loc[mask_b, feature_cols_b].dropna()
print(f"BFP-B running samples: {len(df_b):,}")

# BFP-B runs Jun-Aug 20. Use its first 2 weeks as baseline (Jun 1-14).
healthy_mask_b = df_b.index < "2024-06-15"
X_healthy_b = df_b.loc[healthy_mask_b]
print(f"Healthy baseline samples: {len(X_healthy_b):,}")

scaler_b = StandardScaler()
X_healthy_b_scaled = scaler_b.fit_transform(X_healthy_b)

iso_b = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
iso_b.fit(X_healthy_b_scaled)

X_all_b_scaled = scaler_b.transform(df_b)
df_b["anomaly_score"] = iso_b.decision_function(X_all_b_scaled)
df_b["is_anomaly"] = iso_b.predict(X_all_b_scaled)

n_anomalies_b = (df_b["is_anomaly"] == -1).sum()
print(f"\nBFP-B anomalies detected: {n_anomalies_b} ({n_anomalies_b/len(df_b)*100:.1f}%)")

## Step 6: Visualize Detected Anomalies on Time Series

Overlay the Isolation Forest anomaly detections on the raw sensor data to see whether the model flags the right periods. The known fault windows are:
- **BFP-A seal degradation**: April-June
- **BFP-B bearing wear**: July 15 - August 20
- **BFP-A misalignment**: October 15 - December 31

In [ ]:
# BFP-A: Anomalies overlaid on bearing temperature
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

# Bearing temperature with anomaly markers
anomaly_mask_a = df_a["is_anomaly"] == -1
axes[0].plot(df_a.index, df_a["U1_BFPA_BRG_DE_TEMP"],
             color="#5FCCDB", linewidth=0.4, alpha=0.6, label="DE Bearing Temp")
axes[0].scatter(df_a.index[anomaly_mask_a],
                df_a.loc[anomaly_mask_a, "U1_BFPA_BRG_DE_TEMP"],
                c="red", s=8, zorder=5, label="Anomaly")
axes[0].axhline(y=setpoints["journal_bearing_high_alarm_degc"],
                color="orange", linestyle="--", alpha=0.7, label="Alarm (85 degC)")
axes[0].set_ylabel("Temperature (degC)")
axes[0].set_title("BFP-A: Bearing Temperature with Isolation Forest Anomalies")
axes[0].legend(fontsize=8)

# Anomaly score over time
axes[1].plot(df_a.index, df_a["anomaly_score"], color="#2D6A7A", linewidth=0.5)
axes[1].axhline(y=0, color="red", linestyle="--", alpha=0.5, label="Anomaly boundary")
axes[1].fill_between(df_a.index, df_a["anomaly_score"], 0,
                     where=df_a["anomaly_score"] < 0,
                     color="red", alpha=0.3)
axes[1].set_ylabel("Anomaly Score")
axes[1].set_xlabel("Date")
axes[1].set_title("BFP-A: Isolation Forest Anomaly Score (negative = anomalous)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# BFP-B: Anomalies overlaid on bearing temperature
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

anomaly_mask_b = df_b["is_anomaly"] == -1
axes[0].plot(df_b.index, df_b["U1_BFPB_BRG_NDE_TEMP"],
             color="#5FCCDB", linewidth=0.4, alpha=0.6, label="NDE Bearing Temp")
axes[0].scatter(df_b.index[anomaly_mask_b],
                df_b.loc[anomaly_mask_b, "U1_BFPB_BRG_NDE_TEMP"],
                c="red", s=8, zorder=5, label="Anomaly")
axes[0].axhline(y=setpoints["journal_bearing_high_alarm_degc"],
                color="orange", linestyle="--", alpha=0.7, label="Alarm (85 degC)")
axes[0].set_ylabel("Temperature (degC)")
axes[0].set_title("BFP-B: NDE Bearing Temperature with Isolation Forest Anomalies")
axes[0].legend(fontsize=8)

axes[1].plot(df_b.index, df_b["anomaly_score"], color="#2D6A7A", linewidth=0.5)
axes[1].axhline(y=0, color="red", linestyle="--", alpha=0.5, label="Anomaly boundary")
axes[1].fill_between(df_b.index, df_b["anomaly_score"], 0,
                     where=df_b["anomaly_score"] < 0,
                     color="red", alpha=0.3)
axes[1].set_ylabel("Anomaly Score")
axes[1].set_xlabel("Date")
axes[1].set_title("BFP-B: Isolation Forest Anomaly Score")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Step 7: Health Dashboard Summary

Aggregate the anomaly detections into a dashboard-style summary. For each pump and each sensor group, identify the first date anomalies started appearing and the total count. This is the kind of output that would feed into a control room monitoring display or a weekly maintenance report.

In [ ]:
# Build health dashboard
dashboard_rows = []

# BFP-A anomalies by month
for month in range(1, 13):
    month_mask = df_a.index.month == month
    month_data = df_a.loc[month_mask]
    if len(month_data) == 0:
        continue
    n_anom = (month_data["is_anomaly"] == -1).sum()
    pct = n_anom / len(month_data) * 100
    dashboard_rows.append({
        "Pump": "BFP-A",
        "Month": pd.Timestamp(2024, month, 1).strftime("%b"),
        "Running Hours": len(month_data),
        "Anomaly Hours": n_anom,
        "Anomaly %": round(pct, 1),
    })

# BFP-B anomalies by month
for month in range(1, 13):
    month_mask = df_b.index.month == month
    month_data = df_b.loc[month_mask]
    if len(month_data) == 0:
        continue
    n_anom = (month_data["is_anomaly"] == -1).sum()
    pct = n_anom / len(month_data) * 100
    dashboard_rows.append({
        "Pump": "BFP-B",
        "Month": pd.Timestamp(2024, month, 1).strftime("%b"),
        "Running Hours": len(month_data),
        "Anomaly Hours": n_anom,
        "Anomaly %": round(pct, 1),
    })

dashboard = pd.DataFrame(dashboard_rows)
print("BFP Health Dashboard: Monthly Anomaly Summary")
print("=" * 60)
print(dashboard.to_string(index=False))

# First anomaly detection dates
first_anom_a = df_a.loc[df_a["is_anomaly"] == -1].index.min()
first_anom_b = df_b.loc[df_b["is_anomaly"] == -1].index.min()
print(f"\nFirst anomaly detected:")
print(f"  BFP-A: {first_anom_a}")
print(f"  BFP-B: {first_anom_b}")

In [ ]:
# Visualize dashboard as a heatmap
pivot_a = dashboard[dashboard["Pump"] == "BFP-A"].set_index("Month")["Anomaly %"]
pivot_b = dashboard[dashboard["Pump"] == "BFP-B"].set_index("Month")["Anomaly %"]

# Combine into a matrix
all_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
heat_data = pd.DataFrame(index=["BFP-A", "BFP-B"], columns=all_months, dtype=float)
for m in all_months:
    heat_data.loc["BFP-A", m] = pivot_a.get(m, np.nan)
    heat_data.loc["BFP-B", m] = pivot_b.get(m, np.nan)

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(heat_data.astype(float), annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=1, linecolor="white", ax=ax,
            cbar_kws={"label": "Anomaly %"})
ax.set_title("BFP Health Dashboard: Monthly Anomaly Rate (%)")
ax.set_ylabel("")

# Mark non-running periods
for i, m in enumerate(all_months):
    if pd.isna(heat_data.loc["BFP-A", m]):
        ax.text(i + 0.5, 0.5, "OFF", ha="center", va="center",
                fontsize=8, color="gray")
    if pd.isna(heat_data.loc["BFP-B", m]):
        ax.text(i + 0.5, 1.5, "OFF", ha="center", va="center",
                fontsize=8, color="gray")

plt.tight_layout()
plt.show()

## Key Terms Glossary

| Term | Definition |
|------|-----------|
| **BFP** | Boiler Feed Pump. High-pressure pump that pushes feedwater from the deaerator into the HRSG drum. |
| **DE / NDE** | Drive-End / Non-Drive-End. The two ends of a rotating machine, referring to the coupling side and free end. |
| **Journal Bearing** | A plain bearing that supports the shaft radially. Temperature rise indicates lubrication failure or increased friction. |
| **Thrust Bearing** | A bearing that constrains axial movement of the shaft. Temperature rise indicates axial load changes or wear. |
| **Shaft Vibration** | Displacement of the shaft measured by proximity probes (X and Y). High vibration indicates imbalance, misalignment, or bearing wear. |
| **Isolation Forest** | An unsupervised anomaly detection algorithm that isolates anomalies by random partitioning. Points that require fewer partitions to isolate are more anomalous. |
| **Rolling Statistics** | Mean and standard deviation computed over a sliding time window. Used to smooth noise and detect trends. |
| **Anomaly Score** | A numerical score from the Isolation Forest. Negative values indicate anomalies; more negative means more anomalous. |

## Next Steps

- **Guide 18**: Correlate BFP parameters with unit load to build predictive models
- **Guide 19**: Use multi-class classification to distinguish between fault types (seal, bearing, misalignment)
- **Guide 20**: Build a digital twin using OEM pump curves to track performance degradation